In [1]:
import sys
import os
sys.path.append(os.path.abspath(".."))

In [2]:
from jubilee_controller import JubileeMotionController
import matplotlib.pyplot as plt
import time
import numpy as np 
from pynput import keyboard
import cv2
import lgpio
from datetime import datetime
from camera_controller import camera_tool
from email_sender import send_email
from image_processing import detect_circle
from gripper_controller import Gripper
from micropipette_controller import Micropipette
from magnetic_stirrer_controll import Agitador

In [34]:
jubilee = JubileeMotionController()

In [35]:
jubilee.home_all(mesh_mode_z=False)
jubilee.protect_tools(on=True)
jubilee.move_xyz_absolute(z=50)

In [ ]:
class Micropipette:
    """
    Classe para controle de uma micropipeta automatizada integrada
    em uma Jubilee.

    Esta versão segue a mesma lógica das outras ferramentas (ex: camera_tool, Gripper),
    incluindo controle de estado, nome, e verificação de ferramenta atual.
    """

    def __init__(self, machine, parking_position_xy=(138, 18), move_velocity=3000,linear_coeficientes_ab=(3.49009,12.82974)):
        """
        Inicializa a micropipeta e posiciona o eixo Z em uma altura segura.

        Args:
            machine: Instância de controle da máquina Jubilee.
            parking_position_xy (tuple, optional): Coordenadas (X, Y) para 
                estacionamento da micropipeta. Default é (138, 18).
            move_velocity (int, optional): Velocidade padrão de movimentação (mm/min).
        """
        self.name = "Micropipeta"
        self.parking_position_x, self.parking_position_y = parking_position_xy
        self.machine = machine
        self.move_velocity = move_velocity
        self.linear_coeficientes_ab=linear_coeficientes_ab
        self.liquid_ul = 0
        self.installed = False

        self.machine.gcode('M98 P"/sys/homev.g"')

    def install(self):
        """
        Instala a micropipeta na máquina Jubilee.

        Move o cabeçote até as coordenadas específicas necessárias
        para acoplar a micropipeta ao sistema.
        """
        
    
        if self.machine.tool is None:
            self.machine.gcode("M208 Z150:300")
            

            if self.machine.position[2] < 150:
                self.machine.move_xyz_absolute(z=150)
            self.machine.protect_tools(on=False)

            self.machine.move_xyz_absolute(y=220, velocity=self.move_velocity)
            self.machine.move_xyz_absolute(x=self.parking_position_x, velocity=self.move_velocity)
            self.machine.gcode("G0 U70")
            self.machine.move_xyz_absolute(y=self.parking_position_y, velocity=self.move_velocity)
            self.machine.gcode("G0 U0")
            self.machine.move_xyz_absolute(y=70, velocity=self.move_velocity)

            if self.machine.mode_protect_tools:
                self.machine.protect_tools(on=True)
                self.machine.gcode("M208 Y80:400")

            self.machine.tool = self.name
            self.installed = True

        else:
            print(f"[{self.name}] Não é possível instalar: desinstale '{self.machine.tool}' primeiro.")


    def uninstall(self, velocity=None):
        """
        Desinstala a micropipeta da máquina Jubilee.

        Move o cabeçote até as coordenadas específicas necessárias
        para desacoplar a micropipeta do sistema.
        """

        if self.machine.tool == self.name:
            v = velocity or self.move_velocity
            self.machine.protect_tools(on=False)

            self.machine.move_xyz_absolute(y=90, velocity=v)
            self.machine.move_xyz_absolute(x=self.parking_position_x, velocity=v)
            self.machine.move_xyz_absolute(y=self.parking_position_y, velocity=v)
            self.machine.gcode("G0 U70")
            self.machine.move_xyz_absolute(y=70, velocity=v)
            self.machine.gcode("G0 U0")

            if self.machine.mode_protect_tools:
                self.machine.protect_tools(on=True)

            self.machine.tool = None
            self.installed = False
            self.machine.gcode("M208 Z0:320")

        else:
            print(f"[{self.name}] Nenhuma micropipeta instalada ou outra ferramenta ativa.")



    def press(self, ul):
        """
        Pressiona o êmbolo da micropipeta para definir o volume a ser aspirado.

        Args:
            ul (float): Volume em microlitros (µL) que se deseja aspirar.
                        O valor máximo permitido é 1200 µL.
        """
        if ul > 1200:
            print(f"[{self.name}] Não tente pipetar mais que 1200 µL.")
            return

        if self.liquid_ul > 0:
            self.machine.gcode("G0 V350 F10000")
            self.liquid_ul = 0

        next_position = round(((ul + self.linear_coeficientes_ab[1])/self.linear_coeficientes_ab[0]),2)
        self.machine.gcode(f"G0 V{next_position} F10000")


    def press_step(self, give_step):
        """Pressiona o êmbolo diretamente em um valor de passo."""
        if give_step > 300:
            print(f"[{self.name}] Limite máximo de curso atingido.")
            return

        if self.liquid_ul > 0:
            self.machine.gcode("G0 V350 F1200")
            self.liquid_ul = 0

        self.machine.gcode(f"G0 V{give_step}")


    def aspirate(self):
        """
        Aspira o líquido previamente configurado pelo método `press`.

        Atualiza o atributo `liquid_ul` com o volume atualmente aspirado.
        """
        positions = self.machine.gcode("M114")
        valor_v = float(positions.split("V:")[1].split("E:")[0].strip())

        if valor_v == 0:
            print(f"[{self.name}] Use o método 'press' antes de aspirar.")
            return

        self.liquid_ul = valor_v * self.linear_coeficientes_ab[0] + self.linear_coeficientes_ab[1]
        self.machine.gcode("G0 V0 F4000")


    def dispense(self,velocity=None):
        """
        Dispensa o líquido atualmente aspirado.
        """
        if velocity == None:
            self.machine.gcode("G0 V352 F800")
        else:
            velocidade_step_min=(velocity * 60) / self.linear_coeficientes_ab[0]
            self.machine.gcode(f"G0 V352 F{int(velocidade_step_min)}")
            
        self.liquid_ul = 0
        self.machine.gcode("G0 V175 F10000")


    def eject_tip(self,velocity=9000):
        """
        Ejeta a ponteira da micropipeta.
        """
        self.machine.gcode(f"G0 V450 F{velocity}")
        self.liquid_ul = 0
        self.machine.gcode(f"G0 V0 F{velocity}")


In [80]:
pipeta = Micropipette(jubilee,parking_position_xy=(136.5,16),move_velocity=3000)

In [81]:
pipeta.install()

In [38]:
agitador = Agitador()

error: 'GPIO busy'

In [31]:
jubilee.protect_tools(True)

In [83]:
jubilee.gcode("M208 Z0:320")

''

In [57]:
agitador.desligar()

error: 'unknown handle'

In [54]:
altura_segura_1 = 320
altura_segura_2 = 320
velocidade_movimento_xy = 6000
posicao_descarte = (105,240,270)

In [86]:
def acoplar_ponteira(altura_segura, posicao_caixa_ponteiras):
    jubilee.move_xyz_absolute(z=altura_segura)
    jubilee.move_xyz_absolute(posicao_caixa_ponteiras[0], posicao_caixa_ponteiras[1], velocity=velocidade_movimento_xy)
    jubilee.move_xyz_absolute(z=posicao_caixa_ponteiras[2], velocity=500)
    jubilee.move_xyz_absolute(z=altura_segura)
    posicao_caixa_ponteiras[0] -= 10



def descarte_ponteira(altura_segura,posicao_descarte):
    jubilee.move_xyz_absolute(z=altura_segura)
    jubilee.move_xyz_absolute(posicao_descarte[0], posicao_descarte[1], velocity=velocidade_movimento_xy)
    jubilee.move_xyz_absolute(z=posicao_descarte[2], velocity=1000)
    pipeta.eject_tip()


def pipetar_liquido(origem, destino, volume_ul, altura_segura,velocidade_movimento_xy_z, velocidade_dispensacao=300):
    ciclos = int(volume_ul // 1000)
    restante = volume_ul % 1000

    def transferir(volume):
        jubilee.move_xyz_absolute(z=altura_segura,velocity = velocidade_movimento_xy_z[1])
        pipeta.press(volume)
        jubilee.move_xyz_absolute(origem[0], origem[1], velocity=velocidade_movimento_xy_z[0])
        jubilee.move_xyz_absolute(z=origem[2], velocity=velocidade_movimento_xy_z[1])
        pipeta.aspirate()
        jubilee.move_xyz_absolute(z=altura_segura)
        jubilee.move_xyz_absolute(destino[0], destino[1], velocity=velocidade_movimento_xy_z[0])
        jubilee.move_xyz_absolute(z=destino[2],velocity=velocidade_movimento_xy_z[0])
        pipeta.dispense(velocity=velocidade_dispensacao)

    for _ in range(ciclos):
        transferir(1000)

    if restante > 0:
        transferir(restante)


<p align='justify'>
Inicialmente, 1 mL de nitratos de lantanídeos na proporção 1:0,3 (Gd:Eu), 
com concentrações de 0,2 M de 
<em>Gd(NO<sub>3</sub>)<sub>3</sub>·6H<sub>2</sub>O</em> e 0,06 M de 
<em>Eu(NO<sub>3</sub>)<sub>3</sub>·6H<sub>2</sub>O</em> 
(Sigma-Aldrich), foram adicionados a um balão de fundo redondo acoplado a um condensador. 
Em seguida, 2 mL de 
<em>citrato de sódio</em> (Citrato de Sódio Tribásico Anidro PA, Sigma-Aldrich) 
0,2 M foram adicionados gradualmente ao longo de 3 minutos, sob agitação magnética. 
Após a estabilização da temperatura do sistema a 90&nbsp;°C, utilizando banho de glicerina, 
4 mL de 
<em>NaF</em> (Sigma-Aldrich) 1 M foram adicionados ao sistema, 
que foi mantido sob agitação contínua e temperatura constante de 90&nbsp;°C por 2 horas.
</p>


In [ ]:
#jubilee.move_xyz_absolute(243,161,230)

In [91]:
jubilee.move_xyz_absolute(z=320,velocity=1600)


In [76]:
posicao_balao = (267,161,305)
posicao_caixa_ponteiras = [144.9,312,160]

nitrato_lantanideos = (31,191,171)
#citrato_sodio = (141.0,131.0,153.00)
citrato_sodio = (267,281.0,238.00)



In [56]:
agitador.ligar()

#Acoplar Primeira Ponteira
acoplar_ponteira(altura_segura=altura_segura_1,posicao_caixa_ponteiras=posicao_caixa_ponteiras)

#Pipeta 1000ul de ácido cloroáurico
pipetar_liquido(nitrato_lantanideos,posicao_balao,1000,altura_segura_2,velocidade_movimento_xy_z=[velocidade_movimento_xy,600],velocidade_dispensacao=300)

#Remover ponteira
descarte_ponteira(altura_segura_1,posicao_descarte)

#Acoplar Segunda Ponteira
acoplar_ponteira(altura_segura=altura_segura_1,posicao_caixa_ponteiras=posicao_caixa_ponteiras)

#Pipeta 500ul de borohidreto de sódio
pipetar_liquido(citrato_sodio,posicao_balao,1000,altura_segura_2,velocidade_movimento_xy_z=[10000,1600],velocidade_dispensacao=13)
pipetar_liquido(citrato_sodio,posicao_balao,1000,altura_segura_2,velocidade_movimento_xy_z=[10000,1600],velocidade_dispensacao=13)

#Remover ponteira
descarte_ponteira(altura_segura_1,posicao_descarte)
time.sleep(600)

#Desativar Agitador
agitador.desligar()
agitador.fechar()



Agitador ligado.
Agitador desligado.
Chip GPIO liberado.


In [85]:
pipetar_liquido(citrato_sodio,posicao_balao,2000,altura_segura_2,velocidade_movimento_xy_z=[18000,1600],velocidade_dispensacao=13)